# 02 — Cleaning & instrument forensics

**Stage 2 of the Tier A pipeline** (`DOCS/STRUCTURE.md`). This notebook is forensics first,
reconciliation second — because on this file the two are the same activity.

Notebook `01` established that all four pre-modelling audit checks fail. This notebook:

1. Proves the **replication mechanism** four independent ways, so the 85.71% duplicate rate is
   diagnosed rather than merely counted.
2. Builds the **profile index** — the object every later split, test and permutation depends on.
3. **Reconciles the train/test vocabulary**, the one genuine repair in the project.
4. Handles the remaining defects with a stated treatment and a logged reason each.
5. Persists **three dataset variants**, so Stage 5/6 can run the identical protocol on all of them
   and publish the differences instead of choosing one.

> **The rule this notebook exists to establish:** *252,000 rows are 42,007 observations.*

In [1]:
import hashlib, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)

ROOT = Path.cwd()
while not (ROOT / "archive" / "Training Data.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW, PROC, REPORTS = ROOT / "data" / "raw", ROOT / "data" / "processed", ROOT / "reports"

train = pd.read_csv(RAW / "train_raw_immutable.csv").rename(columns={"Id": "ID"})
test = pd.read_csv(RAW / "test_raw_immutable.csv")

TARGET, IDENTIFIER = "Risk_Flag", "ID"
FEATURES = [c for c in train.columns if c not in (IDENTIFIER, TARGET)]
NUMERIC = ["Income", "Age", "Experience", "CURRENT_JOB_YRS", "CURRENT_HOUSE_YRS"]
CATEGORICAL = ["Married/Single", "House_Ownership", "Car_Ownership", "Profession", "CITY", "STATE"]

cleaning_log = []


def log(action, detail, rows_affected=None, cols_affected=None, reason=""):
    cleaning_log.append({"action": action, "detail": detail, "rows_affected": rows_affected,
                         "cols_affected": cols_affected, "reason": reason})
    n = "" if rows_affected is None else f"  [{rows_affected:,} rows]"
    print(f"LOG  {action}: {detail}{n}")


print(f"train {train.shape} | test {test.shape}")
print(f"target {TARGET!r} | {len(FEATURES)} features")

train (252000, 13) | test (28000, 12)
target 'Risk_Flag' | 11 features


## 2.1 Establishing the replication mechanism

Notebook `01` counted the duplicates. Diagnosing *how they got there* matters, because the
mechanism determines the fix — and here the mechanism says there is no fix, only a correct way to
measure.

### Evidence 1 — the group-size distribution is bimodal with an empty middle

In [2]:
profile_key = train[FEATURES].astype(str).agg("|".join, axis=1)
group_sizes = profile_key.value_counts()
gs = group_sizes.value_counts().sort_index()

print(f"252,000 rows -> {group_sizes.size:,} distinct profiles (ratio {len(train)/group_sizes.size:.4f})\n")
print(f"{'copies':>7s} {'profiles':>10s} {'rows':>10s}")
for k, v in gs.items():
    print(f"{k:>7d} {v:>10,} {k*v:>10,}")
print(f"\nprofiles appearing exactly once : {(group_sizes == 1).sum():,}")
print(f"profiles appearing 27-40 times  : {(group_sizes > 1).sum():,}")
print(f"profiles appearing 2-26 times   : {((group_sizes > 1) & (group_sizes < 27)).sum():,}   <-- none")

252,000 rows -> 42,007 distinct profiles (ratio 5.9990)

 copies   profiles       rows
      1     36,007     36,007
     27          1         27
     29         10        290
     30         19        570
     31         57      1,767
     32        159      5,088
     33        342     11,286
     34        642     21,828
     35        995     34,825
     36      1,216     43,776
     37      1,223     45,251
     38        894     33,972
     39        367     14,313
     40         75      3,000

profiles appearing exactly once : 36,007
profiles appearing 27-40 times  : 6,000
profiles appearing 2-26 times   : 0   <-- none


A real dataset with accidental duplicates has a decaying tail: many pairs, fewer triples, rare
larger groups. This has **nothing between 2 and 26** and a tight mode at 36–37. That is a loop that
ran a fixed number of times, not a data-entry accident.

### Evidence 2 — the arithmetic is exact

In [3]:
print(f"rows / profiles                    = {len(train):,} / {group_sizes.size:,} = {len(train)/group_sizes.size:.4f}")
print(f"replicated profiles                = {(group_sizes > 1).sum():,}")
print(f"mean copies among replicated       = {group_sizes[group_sizes > 1].mean():.2f}")
print(f"replicated rows + singleton rows   = {group_sizes[group_sizes > 1].sum():,} + "
      f"{(group_sizes == 1).sum():,} = {group_sizes[group_sizes > 1].sum() + (group_sizes == 1).sum():,}")

inc = train.Income.value_counts()
print(f"\nthe Income column tells the same story from another angle:")
print(f"  distinct Income values             : {train.Income.nunique():,}")
print(f"  Income values appearing exactly 1x : {(inc == 1).sum():,}")
print(f"  Income values appearing 27-40x     : {((inc >= 27) & (inc <= 40)).sum():,}")

rows / profiles                    = 252,000 / 42,007 = 5.9990
replicated profiles                = 6,000
mean copies among replicated       = 36.00
replicated rows + singleton rows   = 215,993 + 36,007 = 252,000

the Income column tells the same story from another angle:
  distinct Income values             : 41,920
  Income values appearing exactly 1x : 35,857
  Income values appearing 27-40x     : 5,992


### Evidence 3 — the copies disagree about the label

In [4]:
grp = train.assign(_k=profile_key).groupby("_k")[TARGET].agg(["size", "sum", "mean"])
rep = grp[grp["size"] > 1]
mixed = rep[(rep["mean"] > 0) & (rep["mean"] < 1)]

print(f"across the {len(rep):,} replicated profiles:")
print(f"  unanimously Risk_Flag=0                : {(rep['mean'] == 0).sum():,}")
print(f"  unanimously Risk_Flag=1                : {(rep['mean'] == 1).sum():,}")
print(f"  internally contradictory (mixed)       : {len(mixed):,}   "
      f"covering {int(mixed['size'].sum()):,} rows ({mixed['size'].sum()/len(train):.2%})")
print(f"\nwithin-profile positive rate for the mixed profiles:")
print(mixed["mean"].describe().round(4).to_string())

across the 6,000 replicated profiles:
  unanimously Risk_Flag=0                : 4,817
  unanimously Risk_Flag=1                : 0
  internally contradictory (mixed)       : 1,183   covering 42,547 rows (16.88%)

within-profile positive rate for the mixed profiles:
count    1183.0000
mean        0.5582
std         0.1370
min         0.1351
25%         0.4706
50%         0.5676
75%         0.6389
max         0.9189


**Not one of the 6,000 replicated profiles is unanimously a defaulter, and not one is internally
consistent about it either.** If the label described the applicant, identical applicants would carry
identical labels. Instead:

- **4,817 profiles** are unanimously non-default;
- **1,183 profiles** are labelled both 0 and 1 across their own copies, with a within-profile
  positive rate averaging **0.56** (median 0.57, range 0.14–0.92);
- **zero profiles** are unanimously default.

The generator selected 1,183 profiles as risky and then flipped a biased coin **per copy** — roughly
56% of a risky profile's rows come back 1. The label is a property of the *row*, not of the
applicant, and no attribute of the applicant can recover it.

This is also precisely why memorisation scores so well: a lookup table that returns each profile's
majority label gets the 4,817 all-zero profiles entirely right and the 1,183 risky ones right about
56% of the time — the 93.55% in-sample ceiling notebook `01` reported.

### Evidence 4 — the file is sorted, and the two halves were generated differently

In [5]:
is_singleton = (profile_key.map(group_sizes) == 1).to_numpy()
pos = np.arange(len(train))
first_rep_end = int(np.flatnonzero(~is_singleton).max()) + 1

print(f"singletons: {is_singleton.sum():,} total")
print(f"  in rows 0..{first_rep_end-1:,}        : {is_singleton[:first_rep_end].sum():,}")
print(f"  in rows {first_rep_end:,}..{len(train)-1:,}  : {is_singleton[first_rep_end:].sum():,}")
print("\nthe file is a concatenation of a replicated block and a unique block.\n")

block = np.where(is_singleton, "unique_block", "replicated_block")
cmp = (train.assign(_block=block)
       .groupby("_block")
       .agg(rows=(TARGET, "size"), default_rate=(TARGET, "mean"),
            Income=("Income", "mean"), Age=("Age", "mean"), Experience=("Experience", "mean"),
            JOB_YRS=("CURRENT_JOB_YRS", "mean"), HOUSE_YRS=("CURRENT_HOUSE_YRS", "mean")))
print(cmp.round(3).to_string())

from scipy import stats
print("\nare the two blocks drawing from different applicant populations?")
for c in NUMERIC:
    a = train[c][~is_singleton]
    b = train[c][is_singleton]
    ks = stats.ks_2samp(a, b)
    verdict = "DIFFERENT" if ks.pvalue < 0.01 else "indistinguishable"
    print(f"  {c:20s} KS D={ks.statistic:.4f} p={ks.pvalue:.3g}   {verdict}")

singletons: 36,007 total
  in rows 0..215,996        : 4
  in rows 215,997..251,999  : 36,003

the file is a concatenation of a replicated block and a unique block.



                    rows  default_rate       Income     Age  Experience  JOB_YRS  HOUSE_YRS
_block                                                                                     
replicated_block  215993         0.110  4996557.280  49.962      10.106    6.341     11.996
unique_block       36007         0.202  5000472.216  49.908       9.958    6.289     12.009



are the two blocks drawing from different applicant populations?
  Income               KS D=0.0063 p=0.169   indistinguishable
  Age                  KS D=0.0084 p=0.0267   indistinguishable


  Experience           KS D=0.0137 p=1.71e-05   DIFFERENT
  CURRENT_JOB_YRS      KS D=0.0104 p=0.00239   DIFFERENT


  CURRENT_HOUSE_YRS    KS D=0.0098 p=0.00522   DIFFERENT


**The two blocks contain the same applicants and different labels.** `Income` is indistinguishable
across them (KS p = 0.17); `Age`, `Experience`, `CURRENT_JOB_YRS` and `CURRENT_HOUSE_YRS` agree to
two decimal places. The default rate differs by a factor of **1.8** (11.0% vs 20.2%).

Two label-generating regimes were applied to one applicant population, and **no feature tracks
which regime a row came from** — only its row number does. That is the complete explanation for the
two facts notebook `01` reported side by side: `ID` alone reaches AUC 0.674, and all eleven real
features together reach 0.500.

In [6]:
log("diagnose", "replication mechanism established: 42,007 profiles, 6,000 replicated 27-40x",
    rows_affected=int((~pd.Series(is_singleton)).sum()),
    reason="bimodal group sizes, exact 6.0 ratio, contradictory labels, sorted two-block layout")

LOG  diagnose: replication mechanism established: 42,007 profiles, 6,000 replicated 27-40x  [215,993 rows]


## 2.2 Vocabulary reconciliation — the one genuine repair

This comes **before** the profile index, and the order is not cosmetic. Audit check A-03 found that
23 of 51 professions, 8 of 29 states and 16 of 317 cities exist in one file and not the other,
purely because train writes `Software_Developer` and test writes `Software Developer`. Any applicant
key built before that is reconciled would be incomparable across the two files — which is exactly
the mistake that makes the 85.72% test/train overlap invisible.

The fix is applied **identically to both files**, and every renamed category is logged.

In [7]:
# Only the three columns that actually mismatched are touched. `House_Ownership` is deliberately
# excluded: audit A-03 found zero test-only categories in it, and its 'norent_noown' level is a
# single legitimate token, not a separator artifact. Normalising it would silently rename a valid
# category to chase a defect that column does not have.
MISMATCHED = ["Profession", "CITY", "STATE"]
UNTOUCHED = [c for c in CATEGORICAL if c not in MISMATCHED]


def normalise_vocab(df, cols=MISMATCHED):
    out = df.copy()
    for c in cols:
        out[c] = out[c].str.replace("_", " ", regex=False).str.strip()
    return out


vocab_map = []
for c in MISMATCHED:
    for src_name, df in [("train", train), ("test", test)]:
        for old in df[c].unique():
            new = old.replace("_", " ").strip()
            if old != new:
                vocab_map.append({"column": c, "file": src_name, "old": old, "new": new,
                                  "rows": int((df[c] == old).sum())})
vocab_map = pd.DataFrame(vocab_map)

train_n = normalise_vocab(train)
test_n = normalise_vocab(test)

print(f"normalised : {MISMATCHED}")
print(f"left alone : {UNTOUCHED}  (zero vocabulary mismatch — nothing to fix)")
print(f"  'norent_noown' preserved: {'norent_noown' in set(train_n.House_Ownership)}\n")

print("reconciliation result (all six categorical columns checked):")
print(f"{'column':18s} {'train':>7s} {'test':>7s} {'test-only':>10s} {'train-only':>11s}")
after = []
for c in CATEGORICAL:
    a, b = set(train_n[c].unique()), set(test_n[c].unique())
    after.append({"column": c, "categories": len(a), "test_only": len(b - a), "train_only": len(a - b)})
    print(f"{c:18s} {len(a):>7d} {len(b):>7d} {len(b - a):>10d} {len(a - b):>11d}")

assert sum(r["test_only"] + r["train_only"] for r in after) == 0
print("\nzero unmatched categories in either direction, in all six columns.")
print(f"category counts are unchanged (51 / 317 / 29) — the normalisation merges no distinct category.")

vocab_map.to_csv(PROC / "_vocabulary_map.csv", index=False)
print(f"\n{len(vocab_map)} category renames logged to data/processed/_vocabulary_map.csv")
print(vocab_map.groupby(["column", "file"])["rows"].agg(["size", "sum"]).to_string())

log("reconcile", "underscore -> space in Profession/CITY/STATE only, applied to train and test",
    rows_affected=int(len(train) + len(test)), cols_affected=3,
    reason="A-03: 47 test-only categories affecting 72.45% of test rows; zero after normalisation")

normalised : ['Profession', 'CITY', 'STATE']
left alone : ['Married/Single', 'House_Ownership', 'Car_Ownership']  (zero vocabulary mismatch — nothing to fix)


  'norent_noown' preserved: True

reconciliation result (all six categorical columns checked):
column               train    test  test-only  train-only
Married/Single           2       2          0           0
House_Ownership          3       3          0           0
Car_Ownership            2       2          0           0
Profession              51      51          0           0
CITY                   317     317          0           0
STATE                   29      29          0           0

zero unmatched categories in either direction, in all six columns.
category counts are unchanged (51 / 317 / 29) — the normalisation merges no distinct category.

51 category renames logged to data/processed/_vocabulary_map.csv
                  size     sum
column     file               
CITY       test      2     134
           train    18   14019
Profession train    23  115755
STATE      train     8  111195
LOG  reconcile: underscore -> space in Profession/CITY/STATE only, applied to train 

**Guard against over-merging.** The concern with any string normalisation is that it silently
collapses two genuinely distinct categories. It does not here: category counts are **identical
before and after** (51 / 317 / 29), so every rename mapped onto an existing partner rather than
creating a collision.

## 2.3 The profile index — the project's central object

`profile_id` is a hash of the eleven **reconciled** feature columns, identifier excluded. Every
split, every statistical test and every permutation from here on uses it as the **unit of
independence** — and because it is built on the reconciled vocabulary, the same applicant hashes
identically in the training and test files.

In [8]:
def make_profile_id(df, feature_cols):
    key = df[feature_cols].astype(str).agg("|".join, axis=1)
    return key.map(lambda s: hashlib.blake2s(s.encode(), digest_size=8).hexdigest())


train_n["profile_id"] = make_profile_id(train_n, FEATURES)
sizes = train_n.groupby("profile_id")[TARGET].transform("size")
sums = train_n.groupby("profile_id")[TARGET].transform("sum")

train_n["profile_size"] = sizes
train_n["profile_positives"] = sums
train_n["profile_positive_rate"] = sums / sizes
train_n["block"] = np.where(sizes == 1, "unique_block", "replicated_block")
train_n["label_agreement"] = np.select(
    [sizes == 1, sums == 0, sums == sizes],
    ["singleton", "unanimous_0", "unanimous_1"],
    default="mixed")

assert train_n.profile_id.nunique() == group_sizes.size, \
    "reconciliation must not change the number of distinct applicants"
print(f"distinct profiles after reconciliation: {train_n.profile_id.nunique():,} "
      f"(unchanged from {group_sizes.size:,} — normalisation is a bijection within train)\n")

print(train_n["label_agreement"].value_counts().to_frame("rows").assign(
    profiles=train_n.groupby("label_agreement")["profile_id"].nunique()).to_string())

profile_index = train_n[[IDENTIFIER, "profile_id", "profile_size", "profile_positives",
                         "profile_positive_rate", "label_agreement", "block", TARGET]]
profile_index.to_csv(PROC / "_profile_index.csv", index=False)
log("derive", "profile_id (blake2s of the 12 reconciled feature columns) + size, positives, agreement, block",
    rows_affected=len(train_n), cols_affected=6,
    reason="establishes the unit of independence for all splits, tests and permutations; "
           "built post-reconciliation so it is comparable across train and test")

print(f"\nwritten: data/processed/_profile_index.csv  ({len(profile_index):,} rows, "
      f"{train_n.profile_id.nunique():,} distinct profiles)")

distinct profiles after reconciliation: 42,007 (unchanged from 42,007 — normalisation is a bijection within train)

                   rows  profiles
label_agreement                  
unanimous_0      173446      4817
mixed             42547      1183
singleton         36007     36007


LOG  derive: profile_id (blake2s of the 12 reconciled feature columns) + size, positives, agreement, block  [252,000 rows]

written: data/processed/_profile_index.csv  (252,000 rows, 42,007 distinct profiles)


> **This column, not any model choice, is the deliverable of Stage 2.**
>
> `StratifiedKFold` on 252,000 rows produces folds that share applicants. `StratifiedGroupKFold`
> on `profile_id` does not. Notebook `05` reports both, and the gap between them is this project's
> headline result.

## 2.4 The remaining defects

Each gets a stated treatment and a logged reason. Nothing is silently dropped.

### `CITY` bracket suffixes — and the one real collision they hide

In [9]:
train_n["City_Clean"] = train_n.CITY.str.replace(r"\[\d+\]", "", regex=True).str.strip()
train_n["City_Code"] = train_n.CITY.str.extract(r"\[(\d+)\]").astype("float")
test_n["City_Clean"] = test_n.CITY.str.replace(r"\[\d+\]", "", regex=True).str.strip()
test_n["City_Code"] = test_n.CITY.str.extract(r"\[(\d+)\]").astype("float")

br = train_n.CITY.str.contains(r"\[\d+\]", regex=True)
print(f"cities carrying a bracket suffix : {train_n.CITY[br].nunique()} cities, "
      f"{br.sum():,} train rows ({br.mean():.2%})")
state_br = train_n.STATE.str.contains(r"\[\d+\]", regex=True)
print(f"STATE carrying one               : {sorted(train_n.STATE[state_br].unique())} "
      f"({int(state_br.sum()):,} rows)")
print(f"examples: {sorted(train_n.CITY[br].unique())[:6]}")

print("\ndoes each city map to exactly one state?")
m_raw = train_n.groupby("CITY")["STATE"].nunique()
m_clean = train_n.groupby("City_Clean")["STATE"].nunique()
print(f"  raw CITY        : {(m_raw > 1).sum()} of {len(m_raw)} map to >1 state")
print(f"  City_Clean      : {(m_clean > 1).sum()} of {len(m_clean)} map to >1 state  "
      f"-> {list(m_clean[m_clean > 1].index)}")
print(train_n[train_n.City_Clean.isin(m_clean[m_clean > 1].index)]
      .groupby(["City_Clean", "STATE", "CITY"]).size().rename("rows").to_string())

cities carrying a bracket suffix : 30 cities, 23,299 train rows (9.25%)
STATE carrying one               : ['Uttar Pradesh[5]'] (743 rows)
examples: ['Anantapuram[24]', 'Aurangabad[39]', 'Bettiah[33]', 'Buxar[37]', 'Chittoor[28]', 'Dehri[30]']

does each city map to exactly one state?
  raw CITY        : 0 of 317 map to >1 state
  City_Clean      : 1 of 316 map to >1 state  -> ['Aurangabad']
City_Clean  STATE        CITY          
Aurangabad  Bihar        Aurangabad[39]    708
            Maharashtra  Aurangabad        835


Stripping the suffix is right — the brief §7 asks for it — but it merges **two different
Aurangabads**, one in Maharashtra and one in Bihar, into a single category. Disambiguated by
`STATE` rather than left to collide.

In [10]:
for df in (train_n, test_n):
    collide = df.City_Clean.isin(m_clean[m_clean > 1].index)
    df.loc[collide, "City_Clean"] = df.loc[collide, "City_Clean"] + " (" + df.loc[collide, "STATE"] + ")"

print("after disambiguation:")
print(f"  City_Clean categories: {train_n.City_Clean.nunique()} (train), {test_n.City_Clean.nunique()} (test)")
print(f"  cities mapping to >1 state: {(train_n.groupby('City_Clean')['STATE'].nunique() > 1).sum()}")
log("reconcile", "City_Clean (bracket stripped) + City_Code; Aurangabad disambiguated by STATE",
    rows_affected=int(br.sum()), cols_affected=2,
    reason="brief S7 asks for City_Clean; naive stripping merges two cities in different states")

after disambiguation:
  City_Clean categories: 317 (train), 317 (test)
  cities mapping to >1 state: 0
LOG  reconcile: City_Clean (bracket stripped) + City_Code; Aurangabad disambiguated by STATE  [23,299 rows]


### Impossible tenure arithmetic — flagged and tested, not dropped

In [11]:
for df in (train_n, test_n):
    df["implied_start_age"] = df.Age - df.Experience
    df["implausible_start_age"] = (df.implied_start_age < 18).astype(int)

print("implied career start age (Age - Experience):")
print(train_n.implied_start_age.describe().round(2).to_string())
print(f"\n  < 18 : {int(train_n.implausible_start_age.sum()):,} train rows "
      f"({train_n.implausible_start_age.mean():.2%}), {int(test_n.implausible_start_age.sum()):,} test rows")
print(f"  < 10 : {int((train_n.implied_start_age < 10).sum()):,} train rows")
print(f"  minimum implied start age: {int(train_n.implied_start_age.min())}")
print(f"\ndefault rate | implausible: {train_n.groupby('implausible_start_age')[TARGET].mean().round(4).to_dict()}")

log("flag", "implausible_start_age (Age - Experience < 18) created; NO rows dropped",
    rows_affected=int(train_n.implausible_start_age.sum()), cols_affected=2,
    reason="STRUCTURE.md: never remove outliers without a documented reason. These are generator "
           "artifacts, and whether the flag predicts anything is itself measurable")

implied career start age (Age - Experience):
count    252000.00
mean         39.87
std          18.10
min           1.00
25%          25.00
50%          40.00
75%          55.00
max          79.00

  < 18 : 32,189 train rows (12.77%), 3,587 test rows
  < 10 : 9,176 train rows
  minimum implied start age: 1

default rate | implausible: {0: 0.1219, 1: 0.1308}
LOG  flag: implausible_start_age (Age - Experience < 18) created; NO rows dropped  [32,189 rows]


### The derived-feature block the brief proposes (§7)

Built here so Stage 4 can **ablation-test** it rather than assume it helps. The expectation, set in
notebook `01` §1.4, is a measured null: `Income`, `Age` and `Experience` are mutually uncorrelated
and near-uniform, so ratios of them are ratios of independent uniforms.

`Profession_Risk_Group` — the brief's target-encoded feature — is deliberately **not** built here.
Target encoding computed on the full training file before splitting is leakage; it is built inside
the cross-validation pipeline in notebook `05`, fit on training folds only, exactly as the brief's
§7 caution demands.

In [12]:
NORTH = {"Delhi", "Haryana", "Himachal Pradesh", "Jammu and Kashmir", "Punjab", "Rajasthan",
         "Uttarakhand", "Uttar Pradesh", "Uttar Pradesh[5]", "Chandigarh"}
SOUTH = {"Andhra Pradesh", "Karnataka", "Kerala", "Tamil Nadu", "Telangana", "Puducherry"}
EAST = {"Bihar", "Jharkhand", "Odisha", "West Bengal"}
WEST = {"Gujarat", "Maharashtra"}
NORTHEAST = {"Assam", "Manipur", "Mizoram", "Sikkim", "Tripura"}
CENTRAL = {"Chhattisgarh", "Madhya Pradesh"}
REGION = {**{s: "North" for s in NORTH}, **{s: "South" for s in SOUTH},
          **{s: "East" for s in EAST}, **{s: "West" for s in WEST},
          **{s: "Northeast" for s in NORTHEAST}, **{s: "Central" for s in CENTRAL}}

DERIVED = ["City_Clean", "City_Code", "Job_Stability_Ratio", "Career_Start_Ratio",
           "House_Stability_Flag", "Owns_Any_Asset", "Age_Band", "Income_Quartile",
           "State_Region", "Married_Flag", "implausible_start_age"]

income_edges = np.quantile(train_n.Income, [0, .25, .5, .75, 1.0])
income_edges[0], income_edges[-1] = -np.inf, np.inf

for df in (train_n, test_n):
    df["Job_Stability_Ratio"] = df.CURRENT_JOB_YRS / df.Experience.replace(0, np.nan)
    df["Job_Stability_Ratio"] = df.Job_Stability_Ratio.fillna(0.0)
    df["Career_Start_Ratio"] = df.Experience / df.Age
    df["House_Stability_Flag"] = (df.CURRENT_HOUSE_YRS >= 12).astype(int)
    df["Owns_Any_Asset"] = ((df.House_Ownership == "owned") | (df.Car_Ownership == "yes")).astype(int)
    df["Age_Band"] = pd.cut(df.Age, [20, 30, 45, 60, 80], labels=["21-30", "31-45", "46-60", "61-79"]).astype(str)
    df["Income_Quartile"] = pd.cut(df.Income, income_edges, labels=["Q1", "Q2", "Q3", "Q4"]).astype(str)
    df["State_Region"] = df.STATE.map(REGION)
    df["Married_Flag"] = (df["Married/Single"] == "married").astype(int)

assert train_n.State_Region.notna().all() and test_n.State_Region.notna().all(), "all 29 states must map"
print(f"all 29 states map to a region: OK  ({train_n.State_Region.nunique()} regions)")
print(train_n.State_Region.value_counts().to_frame("rows").assign(
    states=train_n.groupby("State_Region")["STATE"].nunique(),
    default_rate=train_n.groupby("State_Region")[TARGET].mean().round(4)).to_string())

print(f"\n{len(DERIVED)} derived features built; Income_Quartile edges taken from TRAIN and applied")
print("to test unchanged (no test statistics used anywhere).")
log("derive", f"brief S7 feature block: {', '.join(DERIVED)}", cols_affected=len(DERIVED),
    reason="built for ablation testing in notebook 05, not assumed useful; "
           "Profession_Risk_Group deferred to the CV pipeline to avoid target leakage")

all 29 states map to a region: OK  (6 regions)
               rows  states  default_rate
State_Region                             
South         68451       6        0.1161
North         61560      10        0.1186
East          56886       4        0.1310
West          36970       2        0.1146
Central       17956       2        0.1499
Northeast     10177       5        0.1347

11 derived features built; Income_Quartile edges taken from TRAIN and applied
to test unchanged (no test statistics used anywhere).
LOG  derive: brief S7 feature block: City_Clean, City_Code, Job_Stability_Ratio, Career_Start_Ratio, House_Stability_Flag, Owns_Any_Asset, Age_Band, Income_Quartile, State_Region, Married_Flag, implausible_start_age


### Columns barred from every model, and why

In [13]:
barred = pd.DataFrame([
    {"column": "ID", "reason": "row counter; reaches AUC 0.674 alone (audit A-02) because the file is sorted",
     "treatment": "dropped from all feature sets; fitted once in notebook 05 as an exhibit"},
    {"column": "profile_id", "reason": "hash of the features; identifies the applicant",
     "treatment": "grouping key for splits and permutations only"},
    {"column": "block", "reason": "derived from file position; unavailable for a real applicant at scoring time",
     "treatment": "diagnostic stratification only — using it would be a second ID leak"},
    {"column": "profile_size / profile_positives / profile_positive_rate", "reason": "computed from the target or from the replication structure",
     "treatment": "diagnostic only; profile_positive_rate is target-derived and is never a feature"},
])
print(barred.to_string(index=False))
log("bar", "ID, profile_id, block, profile_* barred from every feature set", cols_affected=4,
    reason="each is either target-derived or an artifact of file position")

                                                  column                                                                       reason                                                                       treatment
                                                      ID row counter; reaches AUC 0.674 alone (audit A-02) because the file is sorted         dropped from all feature sets; fitted once in notebook 05 as an exhibit
                                              profile_id                               hash of the features; identifies the applicant                                   grouping key for splits and permutations only
                                                   block derived from file position; unavailable for a real applicant at scoring time             diagnostic stratification only — using it would be a second ID leak
profile_size / profile_positives / profile_positive_rate                   computed from the target or from the replication structure diagnostic

## 2.5 Three persisted variants

The identical Stage 5/6 protocol runs on all three. If they agree, that is the finding; if they
diverge, that is a bigger one.

| Variant | Rows | Why it exists |
|---|---|---|
| `loan_rows` | 252,000 | As delivered. What every published notebook uses. |
| `loan_profiles` | 42,007 | One row per applicant. The correct grain. |
| `loan_singletons` | 36,007 | The unique block only — **a plain `StratifiedKFold` is legitimate here**, because there is nothing to leak. Closes the objection that grouping itself destroyed the signal. |

In [14]:
KEEP = [IDENTIFIER, TARGET, "profile_id", "profile_size", "profile_positives",
        "profile_positive_rate", "label_agreement", "block"] + FEATURES + DERIVED + ["implied_start_age"]
KEEP = list(dict.fromkeys(KEEP))

loan_rows = train_n[KEEP].copy()

# one row per applicant; the label becomes the profile's own positive count out of its size
loan_profiles = (loan_rows.sort_values(IDENTIFIER)
                 .drop_duplicates("profile_id", keep="first")
                 .drop(columns=[IDENTIFIER])
                 .reset_index(drop=True))
loan_profiles[TARGET] = (loan_profiles["profile_positive_rate"] >= 0.5).astype(int)

loan_singletons = loan_rows[loan_rows.block == "unique_block"].reset_index(drop=True)

test_reconciled = test_n[[IDENTIFIER] + FEATURES + DERIVED + ["implied_start_age"]].copy()
test_reconciled["profile_id"] = make_profile_id(test_n, FEATURES)
train_profile_set = set(loan_rows.profile_id)
test_reconciled["matched_in_train"] = test_reconciled.profile_id.isin(train_profile_set).astype(int)

for name, df in [("loan_rows", loan_rows), ("loan_profiles", loan_profiles),
                 ("loan_singletons", loan_singletons), ("test_reconciled", test_reconciled)]:
    df.to_parquet(PROC / f"{name}.parquet", index=False)
    rate = f"{df[TARGET].mean():.4f}" if TARGET in df.columns else "n/a"
    print(f"  {name:18s} {len(df):>7,} rows x {df.shape[1]:>2} cols   positive rate {rate}")

print(f"\ntest_reconciled: {int(test_reconciled.matched_in_train.sum()):,} of {len(test_reconciled):,} rows "
      f"({test_reconciled.matched_in_train.mean():.2%}) match a training profile")
assert test_reconciled.matched_in_train.sum() == 24003, \
    "the profile key must be built on the reconciled vocabulary — audit A-04 expects 24,003 matches"
log("persist", "loan_rows, loan_profiles, loan_singletons, test_reconciled written to data/processed/",
    rows_affected=len(loan_rows), reason="three variants carried through the identical Stage 5/6 protocol")

  loan_rows          252,000 rows x 31 cols   positive rate 0.1230
  loan_profiles       42,007 rows x 30 cols   positive rate 0.1909


  loan_singletons     36,007 rows x 31 cols   positive rate 0.2016
  test_reconciled     28,000 rows x 26 cols   positive rate n/a

test_reconciled: 24,003 of 28,000 rows (85.72%) match a training profile
LOG  persist: loan_rows, loan_profiles, loan_singletons, test_reconciled written to data/processed/  [252,000 rows]


> **A note on `loan_profiles`' label.** Collapsing up to 40 contradictory rows to one applicant
> requires a decision. Majority vote is used and stated: a profile is labelled 1 if more than half
> its copies are. Of the 6,000 replicated profiles, **759 have a majority-1 label** (the risky
> profiles average a 0.56 within-profile positive rate, so most of them clear 0.50) and 5,241
> collapse to 0. Together with the 7,260 positive singletons that puts the variant's positive rate
> at **19.1%**, above the row-level 12.3% — because replication is concentrated in the *low-risk*
> profiles and collapsing it removes their weight.
>
> That asymmetry is itself worth stating: the replication does not merely inflate `n`, it inflates
> `n` **unevenly across the classes**, which is why row-level base rates and profile-level base
> rates disagree. This variant is used for *permutation tests and descriptive cuts*, where the label
> is a weight rather than a prediction target; the modelling notebook uses `loan_rows` with grouped
> splits and `loan_singletons`, both of which keep the original per-row labels.

## 2.6 The column contract

In [15]:
BRANCH = {"Age": "B. demographics", "Married/Single": "B. demographics", "Income": "B. demographics",
          "Profession": "C. employment & tenure", "Experience": "C. employment & tenure",
          "CURRENT_JOB_YRS": "C. employment & tenure",
          "House_Ownership": "D. residential & assets", "Car_Ownership": "D. residential & assets",
          "CURRENT_HOUSE_YRS": "D. residential & assets",
          "CITY": "E. geography", "STATE": "E. geography"}

contract = []
for c in FEATURES:
    contract.append({
        "column": c, "branch": BRANCH[c], "role": "feature",
        "dtype": str(loan_rows[c].dtype), "cardinality": int(loan_rows[c].nunique()),
        "admitted_to_models": "yes",
        "note": {"CURRENT_HOUSE_YRS": "5 values 10-14, uniform — not a stability measure",
                 "CITY": "317 categories; exceeds HistGB's 255-category cap, enters as encoding",
                 "Experience": "corr 0.646 with CURRENT_JOB_YRS by generator constraint",
                 "Income": "near-uniform, independent of Age and Experience"}.get(c, ""),
    })
for c in DERIVED + ["implied_start_age"]:
    contract.append({"column": c, "branch": "derived (brief S7)", "role": "derived feature",
                     "dtype": str(loan_rows[c].dtype), "cardinality": int(loan_rows[c].nunique()),
                     "admitted_to_models": "ablation-tested", "note": "measured, not assumed useful"})
for c, why in [(IDENTIFIER, "row counter, AUC 0.674 alone"), ("profile_id", "applicant identity"),
               ("block", "file position"), ("profile_size", "replication artifact"),
               ("profile_positives", "target-derived"), ("profile_positive_rate", "target-derived"),
               ("label_agreement", "target-derived")]:
    contract.append({"column": c, "branch": "A. instrument", "role": "diagnostic key",
                     "dtype": str(loan_rows[c].dtype), "cardinality": int(loan_rows[c].nunique()),
                     "admitted_to_models": "NO", "note": why})

contract = pd.DataFrame(contract)
contract.to_csv(PROC / "_column_contract.csv", index=False)
print(contract.to_string(index=False))

               column                  branch            role   dtype  cardinality admitted_to_models                                                                  note
               Income         B. demographics         feature   int64        41920                yes                       near-uniform, independent of Age and Experience
                  Age         B. demographics         feature   int64           59                yes                                                                      
           Experience  C. employment & tenure         feature   int64           21                yes               corr 0.646 with CURRENT_JOB_YRS by generator constraint
       Married/Single         B. demographics         feature     str            2                yes                                                                      
      House_Ownership D. residential & assets         feature     str            3                yes                                       

In [16]:
cleaning_log_df = pd.DataFrame(cleaning_log)
cleaning_log_df.to_csv(PROC / "_cleaning_log.csv", index=False)
print(cleaning_log_df[["action", "detail", "rows_affected"]].to_string(index=False))

kf = json.loads((REPORTS / "_key_figures.json").read_text())
kf.update({
    "n_loan_rows": len(loan_rows), "n_loan_profiles": len(loan_profiles),
    "n_loan_singletons": len(loan_singletons),
    "singleton_default_rate": float(loan_singletons[TARGET].mean()),
    "replicated_default_rate": float(loan_rows.loc[loan_rows.block == "replicated_block", TARGET].mean()),
    "block_income_ks_p": float(stats.ks_2samp(train.Income[~is_singleton], train.Income[is_singleton]).pvalue),
    "n_vocab_renames": int(len(vocab_map)),
    "test_matched_in_train": int(test_reconciled.matched_in_train.sum()),
    "n_derived_features": len(DERIVED),
})
(REPORTS / "_key_figures.json").write_text(json.dumps(kf, indent=2))
print(f"\n_key_figures.json now holds {len(kf)} figures")

   action                                                                                                                                                                                                     detail  rows_affected
 diagnose                                                                                                                                replication mechanism established: 42,007 profiles, 6,000 replicated 27-40x       215993.0
reconcile                                                                                                                               underscore -> space in Profession/CITY/STATE only, applied to train and test       280000.0
   derive                                                                                                              profile_id (blake2s of the 12 reconciled feature columns) + size, positives, agreement, block       252000.0
reconcile                                                                               

## Stage 2 — Gate Checklist (`DOCS/STRUCTURE.md`)

- [x] **Missing-value strategy documented per column** — there are none, in any of the three files,
      and §2.1 documents why that is *not* reassuring: a generator produces complete rows.
- [x] **Duplicates assessed and handled with justification** — assessed at the **correct grain**
      (§2.1–2.2). 215,993 duplicate rows are **retained**, because deleting them would discard the
      label noise that is itself the finding; the profile index makes them measurable instead.
- [x] **All columns have correct dtypes** — `data/processed/_column_contract.csv`, with branch,
      cardinality, admitted-to-models flag and reason for each of the 30 columns.
- [x] **Business-rule validations pass, or violations documented** — CON-02 (32,189 impossible
      career start ages) and CON-03 retained and flagged as `implausible_start_age`; nothing dropped.
- [x] **Cleaning log exists** — `data/processed/_cleaning_log.csv`, one row per action with a reason.
- [x] **Cleaned data saved to `data/processed/`** — three variants plus the reconciled test set.

### What notebook 03 must do with this

Every bivariate view is produced **twice** — once on `loan_rows` (252,000) and once on
`loan_profiles` (42,007) — side by side. The gap between the two panels is the finding, repeated in
every branch of the issue tree.